# FinQA: three training passes, tested after each

Trains Gemma 3 4B on the same 1,026 solved questions as before, three passes, and tests it on the 297 unseen questions after every pass. This gives the accuracy-by-pass curve. About 3 hours in all.

1. **Runtime > Change runtime type > T4 GPU**, save.
2. **Runtime > Run all**. No uploads needed.
3. Keep the tab open. `finqa_pass1.zip`, `finqa_pass2.zip` and `finqa_pass3.zip` download as each pass finishes; send them to Claude.

In [ ]:
%%capture
!pip install -q unsloth
!rm -rf /content/slm-opti && git clone -q --depth 1 -b v2-split https://github.com/eshwarpotturi/slm-opti /content/slm-opti

In [ ]:
import os, sys, json, time, glob, zipfile
os.environ["TZ"] = "Asia/Kolkata"; time.tzset()   # log times in Indian time, shutil
os.chdir("/content/slm-opti"); sys.path.insert(0, "/content/slm-opti"); sys.path.insert(0, "/content/slm-opti/fin")
import torch
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all."
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(model_name="unsloth/gemma-3-4b-it", max_seq_length=3072, load_in_4bit=True, full_finetuning=False)

import stack, run
def _msgs(messages):
    return [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]

@torch.no_grad()
def hf_chat(messages, temperature=0, max_tokens=400):
    inputs = tokenizer.apply_chat_template(_msgs(messages), add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to("cuda")
    kw = dict(max_new_tokens=max_tokens, do_sample=temperature > 0)
    if temperature > 0:
        kw["temperature"] = temperature
    out = model.generate(**inputs, **kw)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

stack.HF_CHAT = hf_chat

In [ ]:
import run
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
pool = [t for t in json.load(open("fin/data/train.json"))
        if (isinstance(t["answer"], str) or (t["formula"] and run._safe(t))) and len(run.render(t).split()) <= 900]
rows = []
for t in pool:
    target = t["answer"] if isinstance(t["answer"], str) else t["formula"]
    msgs = [{"role": "user", "content": [{"type": "text", "text": run.TUNED + "\n\n" + run.ask(t)}]},
            {"role": "assistant", "content": [{"type": "text", "text": target}]}]
    rows.append({"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")})
ds = Dataset.from_list(rows)
print(len(rows), "training examples")
model = FastModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407)

In [ ]:
PASSES = 3          # about 36 minutes of training and 25 minutes of testing per pass
os.makedirs("fin/results", exist_ok=True)
for p in range(1, PASSES + 1):
    model.train()
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=SFTConfig(
        dataset_text_field="text", per_device_train_batch_size=1, gradient_accumulation_steps=8, warmup_steps=5,
        num_train_epochs=1, learning_rate=2e-4, logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407 + p, report_to="none", output_dir="outputs"))
    trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n", response_part="<start_of_turn>model\n")
    t0 = time.time(); trainer.train()
    json.dump({"pass": p, "minutes": round((time.time() - t0) / 60, 1), "examples": len(rows), "log": trainer.state.log_history},
              open("fin/results/finetune_log_pass%d.json" % p, "w"))
    model.save_pretrained("/content/fin_adapter_pass%d" % p)
    model.eval()
    for layers in ["0", "2"]:      # fine-tuned alone, and with the figure check
        run.main(["hf:gemma-3-4b-tuned", "--layers", layers, "--tag", "ep%d" % p, "--tuned", "--workers", "1", "--budget", "100000"])
    with zipfile.ZipFile("/content/finqa_pass%d.zip" % p, "w", zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob("fin/results/*ep%d.jsonl" % p) + ["fin/results/finetune_log_pass%d.json" % p]:
            z.write(f)
    from google.colab import files
    files.download("/content/finqa_pass%d.zip" % p)